# 13 — Caché de imágenes para el modelado

Esta notebook abre la etapa de modelado. Su propósito es reemplazar la lectura de ~105 mil archivos netCDF, que haría lento cada lote de entrenamiento, por un único arreglo en disco que el cargador lee directamente y que puede subirse a la nube.

**Qué contiene el caché**

Un arreglo `uint16` de forma (instante × banda × fila × columna) con los enteros empaquetados del producto CMIP, sin convertir a temperatura de brillo. El recorte espacial es el mínimo que contiene los parches de 32 × 32 píxeles de las 63 estaciones (filas 61–120 y columnas 70–119 de la grilla de 198 × 199). Solo se incluyen los instantes utilizables del inventario de `08-goes_inventory.ipynb`.

**Por qué enteros sin signo y no float16**

Los archivos guardan cada banda como entero de 16 bits sin signo (`_Unsigned = true`) con un `scale_factor` y un `add_offset`. Conservar esos enteros hace el caché exacto: los valores en kelvin se recuperan con la misma aritmética que usa xarray. El float16, en cambio, tiene una resolución de 0.25 K cerca de 250 K, del orden de varias veces el paso de cuantización del sensor.

**Cómo se garantiza la equivalencia**

Durante la construcción se verifica en cada archivo que la codificación (escala, desplazamiento, `_Unsigned`, `_FillValue`) y la grilla lat/lon coincidan con las del primer archivo, y que el recorte no tenga píxeles sin dato. Después, la Sección 2 compara parche por parche el resultado del caché contra `extraer_parche` de `10-goes_patches.ipynb` y exige igualdad exacta, bit a bit.

**Outputs** (en `data/cache/`, fuera del control de versiones)

| Archivo | Contenido |
|---|---|
| `imagenes.npy` | uint16 (N, 5, 60, 50), ~3.1 GB |
| `instantes.parquet` | fila del arreglo → timestamp, satélite, origen, archivo |
| `estaciones.parquet` | estación → fila y columna en la grilla y en el recorte |
| `meta.json` | bandas, recorte, codificación por banda, estado |
| `sha256.txt` | huella del arreglo para verificar la copia en la nube |


In [1]:
import sys, pathlib
_raiz = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / ".gitignore").exists())
sys.path.insert(0, str(_raiz))
import entorno

In [2]:
# Configuración --------------------------------------------------------------------
import hashlib, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

RUTA_BASE  = Path(r"C:\Users\diani\Documents\repositories\nowcasting-bogota")
RUTA_SAT   = RUTA_BASE / "data_ingestion" / "satellite"
RUTA_CACHE = RUTA_BASE / "data" / "cache"

sys.path.append(str(RUTA_BASE / "src"))
import cache_imagenes as ci

SEED       = 42
N_VERIF    = 2000      # instantes al azar para la verificación exacta (× 63 estaciones)

inv = pd.read_parquet(RUTA_SAT / "inventario_goes.parquet")                 # 08
px  = pd.read_csv(RUTA_SAT / "estaciones_pixels.csv")                       # 10
zs  = pd.read_csv(RUTA_SAT / "normalizacion_zscore.csv")                    # 10
assert len(px) == 63 and px["en_margen"].all()
print(f"Inventario: {len(inv):,} instantes | utilizables: {(~inv['excluido_qc']).sum():,}")
print(f"Recorte: {ci.calcular_recorte(px)}")

Inventario: 104,973 instantes | utilizables: 104,745
Recorte: {'f0': 61, 'f1': 121, 'c0': 70, 'c1': 120}


## 1. Construcción

Lee cada archivo utilizable una sola vez y escribe su recorte en el arreglo. Guarda el avance cada 2,000 archivos: si la ejecución se interrumpe, volver a correr la celda continúa desde donde quedó. Si alguna verificación falla (codificación, grilla o píxeles sin dato), la celda se detiene con el nombre del archivo.

In [3]:
t0 = time.time()
meta = ci.construir(inv, px, RUTA_CACHE)
print(f"\nTiempo: {(time.time() - t0) / 60:.1f} min")
print(json.dumps(meta, indent=2, ensure_ascii=False))

Caché (104745, 5, 60, 50) uint16 (3.14 GB) | pendientes 104,745 de 104,745
    2,000 / 104,745  (62 archivos/s, faltan ~28 min)
    4,000 / 104,745  (63 archivos/s, faltan ~27 min)
    6,000 / 104,745  (63 archivos/s, faltan ~26 min)
    8,000 / 104,745  (63 archivos/s, faltan ~26 min)
   10,000 / 104,745  (62 archivos/s, faltan ~25 min)
   12,000 / 104,745  (62 archivos/s, faltan ~25 min)
   14,000 / 104,745  (61 archivos/s, faltan ~25 min)
   16,000 / 104,745  (61 archivos/s, faltan ~24 min)
   18,000 / 104,745  (60 archivos/s, faltan ~24 min)
   20,000 / 104,745  (60 archivos/s, faltan ~23 min)
   22,000 / 104,745  (60 archivos/s, faltan ~23 min)
   24,000 / 104,745  (60 archivos/s, faltan ~22 min)
   26,000 / 104,745  (60 archivos/s, faltan ~22 min)
   28,000 / 104,745  (60 archivos/s, faltan ~21 min)
   30,000 / 104,745  (60 archivos/s, faltan ~21 min)
   32,000 / 104,745  (60 archivos/s, faltan ~20 min)
   34,000 / 104,745  (60 archivos/s, faltan ~20 min)
   36,000 / 104,745  (60

## 2. Verificación de igualdad exacta contra `extraer_parche`

La función se copia sin cambios de `10-goes_patches.ipynb`. Se comparan todos los instantes reconstruidos desde L1b, una muestra aleatoria de instantes de cada satélite y los dos instantes de las figuras de validación de 10. En cada instante se comparan las 63 estaciones. La condición es igualdad exacta (mismo tipo y mismos valores), no una tolerancia.

In [4]:
# extraer_parche — copia literal de 10-goes_patches.ipynb ----------------------------
def extraer_parche(filepath, fila_centro, col_centro, half_size, bandas, zscore_params):
    """Parche (B, 2·half_size, 2·half_size) centrado en (fila_centro, col_centro), normalizado por z-score."""
    f0, f1 = fila_centro - half_size, fila_centro + half_size
    c0, c1 = col_centro - half_size, col_centro + half_size
    with xr.open_dataset(filepath) as ds:
        parche = np.stack([ds[b].values[f0:f1, c0:c1] for b in bandas], axis=0).astype(np.float32)
    assert np.isfinite(parche).all(), f"Parche con píxeles sin dato: {filepath}"
    params = zscore_params.set_index("banda").loc[bandas]
    return (parche - params["mean"].values[:, None, None]) / params["std"].values[:, None, None]

In [5]:
cache = ci.Cache(RUTA_CACHE, zs)
ins = cache.instantes

muestra = pd.concat([
    ins[ins["origen"] == "L1b"],                                                     # los 82
    ins[(ins["origen"] == "CMIP") & (ins["satelite"] == "G16")].sample(N_VERIF // 2, random_state=SEED),
    ins[(ins["origen"] == "CMIP") & (ins["satelite"] == "G19")].sample(N_VERIF // 2, random_state=SEED),
    ins[ins["timestamp"].isin(pd.to_datetime(["2024-02-18 06:30", "2025-04-25 20:00"]))],
]).drop_duplicates("fila")
print(muestra.groupby(["origen", "satelite"]).size().to_string())

distintos, n = [], 0
for r in muestra.itertuples():
    for e in px.itertuples():
        a = extraer_parche(r.filepath, int(e.fila), int(e.col), ci.HALF_SIZE, ci.BANDAS, zs)
        b = cache.parche(r.timestamp, int(e.codigoestacion))
        n += 1
        if not (a.dtype == b.dtype and a.shape == b.shape and np.array_equal(a, b)):
            distintos.append((r.timestamp, e.codigoestacion, float(np.abs(a - b).max())))

print(f"\nParches comparados: {n:,} ({len(muestra):,} instantes × {len(px)} estaciones)")
print(f"Distintos: {len(distintos)}")
assert not distintos, distintos[:10]

origen  satelite
CMIP    G16         1001
        G19         1001
L1b     G16           82

Parches comparados: 131,292 (2,084 instantes × 63 estaciones)
Distintos: 0


## 3. Controles de integridad y huella

In [6]:
img = cache.img
assert cache.meta["completo"] and len(ins) == (~inv["excluido_qc"]).sum() == img.shape[0]
assert ins["timestamp"].is_monotonic_increasing and ins["timestamp"].is_unique
assert set(cache.estaciones["codigoestacion"].astype("int64")) == set(px["codigoestacion"].astype("int64"))
print(f"Forma: {img.shape}  dtype: {img.dtype}  tamaño: {img.nbytes / 1e9:.2f} GB")
print(ins.groupby(["satelite", "origen"]).size().to_string())

# Ninguna fila quedó en cero (fila no escrita) ni con valor de relleno
ceros = np.flatnonzero(np.asarray([(img[i] == 0).all() for i in range(img.shape[0])]))
assert len(ceros) == 0, f"Filas vacías: {ceros[:10]}"

def sha256(ruta, bloque=1 << 24):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        while (b := f.read(bloque)):
            h.update(b)
    return h.hexdigest()

huellas = {n: sha256(RUTA_CACHE / n) for n in ["imagenes.npy", "instantes.parquet", "estaciones.parquet", "meta.json"]}
(RUTA_CACHE / "sha256.txt").write_text("".join(f"{v}  {k}\n" for k, v in huellas.items()))
print((RUTA_CACHE / "sha256.txt").read_text())

Forma: (104745, 5, 60, 50)  dtype: uint16  tamaño: 3.14 GB
satelite  origen
G16       CMIP      65660
          L1b          82
G19       CMIP      39003
a39f1914c6dad85e2724caac509034b0d782cfa51dadc6f25eca493fd51c5326  imagenes.npy
7c8d8af5366c985e470c768a4aec29bb74b46ad24a71928fad8500a0a3ddbb97  instantes.parquet
399578531f38ffce00e666700459f0b5241e8c7302e17b2fba74226fb7d15ea3  estaciones.parquet
6f0048074716e374b885415f774085014d7134b14eca6a1816c1fad590525286  meta.json



## Conclusión

El caché quedó construido y verificado. Contiene los 104,745 instantes utilizables del inventario (65,660 de GOES-16 en CMIP, 82 reconstruidos desde L1b y 39,003 de GOES-19), cada uno con las cinco bandas recortadas a 60 × 50 píxeles, en un arreglo `uint16` de 3.14 GB. La construcción leyó cada archivo una sola vez en 29 minutos, a unos 60 archivos por segundo.

Durante la construcción, los 104,745 archivos superaron las tres comprobaciones: la misma codificación por banda (`scale_factor`, `add_offset`, `_Unsigned = true`, `_FillValue = −1`), la misma grilla lat/lon y ningún píxel sin dato dentro del recorte. Que la codificación sea idéntica en todos los archivos, incluidos los de GOES-19 y los reconstruidos desde L1b, es lo que permite guardar los enteros una sola vez con una única escala por banda.

La verificación comparó 131,292 parches (2,084 instantes × 63 estaciones): los 82 instantes reconstruidos desde L1b, 1,001 de cada satélite y los dos instantes de las figuras de validación de `10-goes_patches.ipynb`. Todos los parches fueron idénticos, en tipo y en valor, a los que produce `extraer_parche`. El cargador puede leer del caché con la garantía de que el modelo recibe exactamente los mismos datos que describe el capítulo 3.

Huella SHA-256 de `imagenes.npy`: `a39f1914c6dad85e2724caac509034b0d782cfa51dadc6f25eca493fd51c5326`. Las huellas de los cuatro archivos quedan en `data/cache/sha256.txt` y sirven para verificar la copia en la nube (S1b).
